# Computer Exercise 13.7 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.7 Minimization — *Exact Trust-Region Subproblem, Negative Curvature & the Hard Case*
> **풀이 일자**: Day 52
> **언어**: Python 3 (NumPy / pandas / Matplotlib)


## 1. 문제 (원문)

> **2.** Solve the trust-region subproblem *exactly* by determining the Levenberg–Marquardt parameter
> $\lambda\ge 0$ for which the solution of $(B+\lambda I)p=-g$ has $\lVert p\rVert=\Delta$. Use the
> eigendecomposition of $B$ to plot the *secular* function $\phi(\lambda)=\lVert p(\lambda)\rVert$,
> handle an **indefinite** model Hessian (negative curvature), and demonstrate the **hard case** in which
> $g$ is orthogonal to the eigenvector of the smallest eigenvalue. Verify your solver against the
> optimality conditions.

### 한국어 풀이용 정리
신뢰영역 부분문제(TRS)를 **정확히** 푼다. 최적해는 어떤 $\lambda\ge0$ 에 대해
$(B+\lambda I)p=-g,\ \lVert p\rVert=\Delta$ 를 만족한다(Levenberg–Marquardt 모수). $B$ 의 고유분해로
**secular 함수** $\phi(\lambda)=\lVert p(\lambda)\rVert$ 를 그리고, **부정부호(음의 곡률)** 헤시안과,
$g$ 가 최소 고윳값의 고유벡터와 직교하는 **hard case** 를 처리한다. 최적성 조건으로 해를 검증한다.


## 2. 수학적 배경

### 2.1 TRS의 최적성 조건 (Moré–Sorensen)
$p^\*$ 가 $\min_{\lVert p\rVert\le\Delta} g^\top p+\tfrac12 p^\top B p$ 의 전역해일 **필요충분조건**은
어떤 $\lambda^\*\ge0$ 가 존재하여
$$
\boxed{\;(B+\lambda^\* I)\,p^\*=-g,\qquad \lambda^\*(\Delta-\lVert p^\*\rVert)=0,\qquad B+\lambda^\* I\succeq 0\;}
$$
을 만족하는 것이다. 즉 내부해면 $\lambda^\*=0$(이때 $B\succeq0$), 경계해면 $\lVert p^\*\rVert=\Delta$.

### 2.2 secular 함수와 고유분해
$B=Q\Lambda Q^\top$, $\Lambda=\mathrm{diag}(\lambda_1\le\cdots\le\lambda_n)$, $\gamma_j=q_j^\top g$ 라 하면
$\lambda>-\lambda_1$ 에서
$$
p(\lambda)=-(B+\lambda I)^{-1}g=-\sum_{j}\frac{\gamma_j}{\lambda_j+\lambda}\,q_j,\qquad
\phi(\lambda)^2=\lVert p(\lambda)\rVert^2=\sum_j\frac{\gamma_j^2}{(\lambda_j+\lambda)^2}.
$$
$\phi$ 는 $(-\lambda_1,\infty)$ 에서 **단조감소**하므로, $\phi(\lambda)=\Delta$ 의 근 $\lambda^\*$ 는 유일하다.
실무에서는 거의 선형인 **secular 방정식** $\,1/\phi(\lambda)-1/\Delta=0\,$ 의 근을 찾는다.

### 2.3 음의 곡률과 hard case
$B$ 가 **부정부호**($\lambda_1<0$)면 뉴턴 스텝은 상승방향일 수 있어 직선탐색 뉴턴이 실패하지만, TRS 해는
$\lambda^\*>-\lambda_1\ge0$ 를 골라 $B+\lambda^\*I\succ0$ 로 만들어 **항상 하강**을 준다.
**Hard case**: $\gamma_1=q_1^\top g=0$ (그리고 $\lambda_1<0$) 이면 $\lambda\to-\lambda_1^+$ 에서도 $\phi$ 가
유한값에 머문다. 그 한계값이 $\Delta$ 보다 작으면 $\lambda$ 만으로는 경계에 닿지 못하므로
$$
p^\*=-\!\!\sum_{\lambda_j\ne\lambda_1}\frac{\gamma_j}{\lambda_j-\lambda_1}q_j+\alpha\,q_1,\qquad
\alpha\ \text{는}\ \lVert p^\*\rVert=\Delta\ \text{가 되도록}
$$
즉 $\lambda^\*=-\lambda_1$ 로 고정하고 **최소 고유벡터 방향** $q_1$ 을 더해 경계를 맞춘다.


## 3. 풀이 흐름

1. **secular 솔버 작성**: 고유분해 $B=Q\Lambda Q^\top$ 로 $\phi(\lambda)$, $p(\lambda)$ 를 구성.
2. **표준(easy) 경우**: 부정부호 $B$, $g$ 일반 — $\phi(\lambda)=\Delta$ 의 근 $\lambda^\*$ 를 이분법으로.
3. **검증**: $(B+\lambda^\*I)p^\*=-g$, $\lVert p^\*\rVert=\Delta$, $B+\lambda^\*I\succeq0$, 모델 감소 $>0$ 확인.
4. **hard case 구성**: $g\perp q_1$ 로 두고 $\phi$ 의 한계값 $<\Delta$ 임을 확인 → $q_1$ 보정으로 $p^\*$ 완성.
5. **시각화 1**: 두 경우의 secular 함수 $\phi(\lambda)$ 와 목표선 $\Delta$, 근 $\lambda^\*$, 수직점근선 $-\lambda_1$.
6. **시각화 2**: 모델 등고선 위에 신뢰영역 원과 정확해 $p^\*$, (참고로) 뉴턴 스텝 방향 표시.
7. **해석**: $\lambda$ 가 곧 모델을 양정치로 "들어올리는" 정칙화 모수임을, hard case의 특이성을 정리.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def trs_exact(B, g, Delta, tol=1e-12):
    # 정확 TRS 해. (lambda*, p*, info) 반환. hard case 처리 포함.
    B = np.asarray(B, float); g = np.asarray(g, float)
    w, Q = np.linalg.eigh(B)          # w ascending eigenvalues
    gamma = Q.T @ g                   # gamma_j = q_j . g
    lam1 = w[0]

    def p_of(lmb):
        return -(Q @ (gamma / (w + lmb)))
    def phi(lmb):
        return np.linalg.norm(gamma / (w + lmb))

    # (i) interior solution possible? lambda=0 needs B PD and ||B^{-1}g||<=Delta
    if lam1 > 0:
        p0 = -np.linalg.solve(B, g)
        if np.linalg.norm(p0) <= Delta:
            return 0.0, p0, "interior (B PD, Newton inside)"

    # active set: smallest eigenvalue components with gamma~0 -> hard-case candidate
    near = np.isclose(w, lam1, atol=1e-12)
    gamma1_zero = np.all(np.abs(gamma[near]) < 1e-12)

    lam_low = max(0.0, -lam1) + 1e-12   # need B+lambda I PD  => lambda > -lam1
    # limiting phi as lambda -> -lam1^+ (only finite if gamma1_zero)
    if gamma1_zero and lam1 < 0:
        # phi at lambda = -lam1 using only non-degenerate terms
        mask = ~near
        phi_lim = np.linalg.norm(gamma[mask] / (w[mask] - lam1))
        if phi_lim < Delta:   # HARD CASE
            lam_star = -lam1
            p_perp = -(Q[:, mask] @ (gamma[mask] / (w[mask] - lam1)))
            # add alpha * q1 to reach the boundary
            q1 = Q[:, 0]
            alpha = np.sqrt(max(Delta**2 - np.dot(p_perp, p_perp), 0.0))
            p_star = p_perp + alpha * q1
            return lam_star, p_star, "HARD CASE (boundary via q1)"

    # (ii) easy boundary case: bisection on phi(lambda)=Delta, lambda in (lam_low, hi)
    lo = lam_low
    hi = lo + 1.0
    while phi(hi) > Delta:
        hi *= 2.0
    for _ in range(200):
        mid = 0.5*(lo+hi)
        if phi(mid) > Delta: lo = mid
        else: hi = mid
        if hi - lo < tol*max(1.0, hi): break
    lam_star = 0.5*(lo+hi)
    return lam_star, p_of(lam_star), "boundary (secular root)"


In [2]:
# ---- Case A: indefinite B, generic g (easy boundary case) ----
B = np.array([[1.0, 0.0],
              [0.0, -2.0]])          # eigenvalues 1, -2  -> INDEFINITE
g = np.array([1.0, 1.0])
Delta = 1.0

lamA, pA, infoA = trs_exact(B, g, Delta)
res = (B + lamA*np.eye(2)) @ pA + g
Bshift_eig = np.linalg.eigvalsh(B + lamA*np.eye(2))
model_red = -(g @ pA + 0.5 * pA @ (B @ pA))

print("=== Case A: indefinite B, generic g ===")
print(f" eig(B)           = {np.linalg.eigvalsh(B)}")
print(f" lambda*          = {lamA:.10f}   ({infoA})")
print(f" p*               = {pA}")
print(f" ||p*||           = {np.linalg.norm(pA):.10f}  (target Delta={Delta})")
print(f" ||(B+lam I)p+g|| = {np.linalg.norm(res):.3e}  (should be ~0)")
print(f" eig(B+lam* I)    = {Bshift_eig}  (should be >= 0)")
print(f" model reduction  = {model_red:.6f}  (>0 means descent)")


=== Case A: indefinite B, generic g ===
 eig(B)           = [-2.  1.]
 lambda*          = 3.0322475511   (boundary (secular root))
 p*               = [-0.24800065 -0.96875987]
 ||p*||           = 1.0000000000  (target Delta=1.0)
 ||(B+lam I)p+g|| = 0.000e+00  (should be ~0)
 eig(B+lam* I)    = [1.03224755 4.03224755]  (should be >= 0)
 model reduction  = 2.124504  (>0 means descent)


In [3]:
# ---- Case B: HARD CASE  (g orthogonal to eigenvector of smallest eigenvalue) ----
# smallest eigenvalue of B is -2 with eigenvector e2 = (0,1); make g perp to it:
g_hard = np.array([1.0, 0.0])        # g . e2 = 0  -> hard case trigger
Delta_h = 1.0

lamB, pB, infoB = trs_exact(B, g_hard, Delta_h)
resB = (B + lamB*np.eye(2)) @ pB + g_hard
model_redB = -(g_hard @ pB + 0.5 * pB @ (B @ pB))

# limiting phi at lambda -> -lam1 (=2) using only the non-degenerate (x) component
phi_lim = abs(g_hard[0]) / (1.0 - (-2.0))   # gamma_x/(lambda_x - lam1) = 1/(1-(-2))=1/3
print("=== Case B: HARD CASE (g perp to min-eigenvector) ===")
print(f" g_hard            = {g_hard}   (orthogonal to e2, the lambda=-2 eigenvector)")
print(f" phi limit @ lam->2 = {phi_lim:.6f}  <  Delta={Delta_h}  -> hard case")
print(f" lambda*           = {lamB:.10f}  = -lambda_min = 2   ({infoB})")
print(f" p*                = {pB}")
print(f" ||p*||            = {np.linalg.norm(pB):.10f}  (target {Delta_h})")
print(f" ||(B+lam I)p+g||  = {np.linalg.norm(resB):.3e}")
print(f" model reduction   = {model_redB:.6f}")


=== Case B: HARD CASE (g perp to min-eigenvector) ===
 g_hard            = [1. 0.]   (orthogonal to e2, the lambda=-2 eigenvector)
 phi limit @ lam->2 = 0.333333  <  Delta=1.0  -> hard case
 lambda*           = 2.0000000000  = -lambda_min = 2   (HARD CASE (boundary via q1))
 p*                = [-0.33333333  0.94280904]
 ||p*||            = 1.0000000000  (target 1.0)
 ||(B+lam I)p+g||  = 0.000e+00
 model reduction   = 1.166667


In [4]:
# ---- summary table ----
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
summary = pd.DataFrame({
    "case":            ["A: indefinite/generic", "B: hard case"],
    "lambda*":         [lamA, lamB],
    "||p*||":          [np.linalg.norm(pA), np.linalg.norm(pB)],
    "min eig(B+lam*I)": [np.linalg.eigvalsh(B+lamA*np.eye(2))[0],
                         np.linalg.eigvalsh(B+lamB*np.eye(2))[0]],
    "model reduction": [model_red, model_redB],
    "type":            [infoA, infoB],
})
summary


,case,lambda*,||p*||,min eig(B+lam*I),model reduction,type
0,A: indefinite/generic,3.032248e+00,1.000000e+00,1.032248e+00,2.124504e+00,boundary (secular root)
1,B: hard case,2.000000e+00,1.000000e+00,0.000000e+00,1.166667e+00,HARD CASE (boundary via q1)


In [5]:
# ---- Visualization 1: secular function phi(lambda) ----
w = np.linalg.eigvalsh(B); lam1 = w[0]   # = -2
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# Case A
lams = np.linspace(-lam1+1e-3, -lam1+8, 600)
Q = np.linalg.eigh(B)[1]
def phi_for(gv, lm):
    gam = Q.T @ gv
    return np.array([np.linalg.norm(gam/(w+l)) for l in lm])
axes[0].plot(lams, phi_for(g, lams), color="tab:blue", lw=1.6, label=r"$\phi(\lambda)=\||p(\lambda)\||$")
axes[0].axhline(Delta, color="k", ls="--", lw=1, label=r"$\Delta$")
axes[0].axvline(-lam1, color="tab:red", ls=":", lw=1, label=r"$-\lambda_{\min}=2$")
axes[0].plot(lamA, Delta, "ro", ms=8, label=r"$\lambda^*$ (root)")
axes[0].set_xlabel(r"$\lambda$"); axes[0].set_ylabel(r"$\phi(\lambda)$")
axes[0].set_title("Case A: secular function (boundary root)")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.5)

# Case B (hard case): phi stays below Delta as lambda -> -lam1
axes[1].plot(lams, phi_for(g_hard, lams), color="tab:green", lw=1.6,
             label=r"$\phi(\lambda)$ (hard case)")
axes[1].axhline(Delta_h, color="k", ls="--", lw=1, label=r"$\Delta$")
axes[1].axhline(1/3, color="tab:purple", ls="-.", lw=1, label=r"limit $\phi\to 1/3<\Delta$")
axes[1].axvline(-lam1, color="tab:red", ls=":", lw=1, label=r"$\lambda^*=-\lambda_{\min}=2$")
axes[1].set_xlabel(r"$\lambda$"); axes[1].set_ylabel(r"$\phi(\lambda)$")
axes[1].set_title("Case B: hard case (no interior root; need $q_1$)")
axes[1].legend(fontsize=8); axes[1].grid(True, ls=":", alpha=0.5)
plt.tight_layout(); plt.savefig("trs_secular.png", dpi=110); plt.show()


In [6]:
# ---- Visualization 2: model contours + trust region + exact step (Case A) ----
xx = np.linspace(-1.6, 1.6, 400); yy = np.linspace(-1.6, 1.6, 400)
XX, YY = np.meshgrid(xx, yy)
MM = g[0]*XX + g[1]*YY + 0.5*(B[0,0]*XX**2 + 2*B[0,1]*XX*YY + B[1,1]*YY**2)

fig, ax = plt.subplots(figsize=(6.4, 6.0))
cs = ax.contour(XX, YY, MM, levels=30, cmap="coolwarm", linewidths=0.7)
th = np.linspace(0, 2*np.pi, 200)
ax.plot(Delta*np.cos(th), Delta*np.sin(th), "k--", lw=1.2, label=r"trust region $\||p\||=\Delta$")
ax.annotate("", xy=(pA[0], pA[1]), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="red", lw=2))
ax.plot(pA[0], pA[1], "ro", ms=8, label=r"exact TRS step $p^*$")
ax.plot(0, 0, "ks", ms=6, label="model center")
ax.set_aspect("equal"); ax.set_xlabel("$p_1$"); ax.set_ylabel("$p_2$")
ax.set_title("Indefinite quadratic model: exact step rides the boundary")
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig("trs_model.png", dpi=110); plt.show()


## 4. 결과 해석

1. **정칙화 모수로서의 $\lambda$**: Case A에서 $B$ 는 부정부호($\lambda_{\min}=-2$)라 뉴턴 스텝은 안장점으로
   향하지만, $\lambda^\*\approx 3.03>2$ 를 더하면 $B+\lambda^\*I\succ0$ 가 되어 정확해 $p^\*$ 가
   **모델을 감소**시키며 신뢰영역 경계에 정확히 놓인다($\lVert p^\*\rVert=\Delta$, 잔차 $\sim10^{-16}$).
2. **secular 함수의 형태**: $\phi(\lambda)$ 는 $\lambda=-\lambda_{\min}$ 에서 수직점근선을 갖고 단조감소하므로,
   $\phi(\lambda)=\Delta$ 의 근은 유일하고 안정적으로 이분법(또는 secular 뉴턴)으로 잡힌다.
3. **Hard case**: $g\perp q_1$ 이면 점근선에서 $\phi$ 가 폭주하지 않고 $1/3<\Delta$ 에 머문다. $\lambda$ 만으로는
   경계에 못 닿으므로 $\lambda^\*=-\lambda_{\min}=2$ 로 고정하고 **최소 고유벡터** $q_1$ 을 $\alpha$ 만큼 더해
   $\lVert p^\*\rVert=\Delta$ 를 맞춘다 — 이 특이성이 정확한 TRS 솔버가 반드시 처리해야 할 부분이다.
4. **dogleg과의 대비**: dogleg(CE 13.7.1)은 $B\succ0$ 를 가정하므로 이런 음의 곡률·hard case를 다룰 수 없다.
   정확 솔버는 비용은 더 들지만($n$ 작을 때 고유분해) **모든 경우에 전역 최적 부분문제 해**를 준다.

### 결론
> **TRS 정확해 = $(B+\lambda I)p=-g$ 의 $\lambda$ 를 secular 방정식으로 찾는 일** —
> $\lambda$ 는 모델을 양정치로 들어올리는 정칙화이며, $g\perp q_{\min}$ 인 hard case에서만
> 최소 고유벡터 보정이 추가로 필요하다.

### 다음 문제 연결
- **CE 13.7.3**: $n$ 이 크면 고유분해가 비싸다. 다음 문제에서 **Steihaug–Toint truncated CG** 로
  TRS를 행렬-벡터 곱만으로 근사해 풀고, 신뢰영역법과 직선탐색법의 총 반복수를 비교한다.
